In [ ]:
# ============================================================
# NETWORK SCIENCE — ASSIGNMENT 3
# QUESTION 1: Configuration Model & Edge-Swapping
# Dataset: Facebook Social Network (SNAP) — ~4000 nodes
# Source: https://snap.stanford.edu/data/ego-Facebook.html
# ============================================================

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import urllib.request
import gzip
import random
from collections import Counter

# ============================================================
# GRAB THE ACTUAL FACEBOOK DATA
# ============================================================
url = "https://snap.stanford.edu/data/facebook_combined.txt.gz"
urllib.request.urlretrieve(url, "facebook.txt.gz")

with gzip.open("facebook.txt.gz", "rt") as f:
    real_g = nx.read_edgelist(f, nodetype=int)

# make sure it's simple and undirected
real_g = nx.Graph(real_g)
print(f"Nodes: {real_g.number_of_nodes()}, Edges: {real_g.number_of_edges()}")

# grab the degree sequence from the real graph
degrees = [d for _, d in real_g.degree()]
degree_hist = Counter(degrees)
max_degree = max(degree_hist.keys())

# ============================================================
# STUB MATCHING: BUILD A GRAPH FROM DEGREES
# ============================================================
def make_graph_from_degrees(degree_sequence):
    """
    Tosses stubs (half-edges) for each node based on its degree,
    shuffles em up, then wires them together pairwise.
    Self-loops and multi-edges get tossed out afterwards.
    """
    stubs = []
    for node, deg in enumerate(degree_sequence):
        stubs.extend([node] * deg)

    # odd stub count? toss one (rare but happens)
    if len(stubs) % 2 != 0:
        stubs.pop()

    random.shuffle(stubs)  # shuffle em up

    g = nx.Graph()
    g.add_nodes_from(range(len(degree_sequence)))

    # pair em up sequentially after the shuffle
    for i in range(0, len(stubs) - 1, 2):
        u, v = stubs[i], stubs[i + 1]
        if u != v:  # no self-loops thanks
            g.add_edge(u, v)

    return g

# ============================================================
# EDGE SWAP: REWIRE WITHOUT LOSING DEGREES
# ============================================================
def rewire_it(g_input, swaps_multiplier=100):
    """
    Takes edges (a,b) and (c,d), swaps to (a,d) and (c,b).
    Keeps going until we've done enough swaps.
    Rejects moves that would create self-loops or duplicate edges.
    """
    g = g_input.copy()
    edges = list(g.edges())
    num_swaps = swaps_multiplier * len(edges)
    swaps_done = 0

    while swaps_done < num_swaps:
        # grab two random edges
        idx1, idx2 = random.sample(range(len(edges)), 2)
        a, b = edges[idx1]
        c, d = edges[idx2]

        # no shared endpoints please (self-loop alert!)
        if len({a, b, c, d}) < 4:
            continue

        # skip if the new edges already exist
        if g.has_edge(a, d) or g.has_edge(c, b):
            continue

        # there it is - do the swap
        g.remove_edge(a, b)
        g.remove_edge(c, d)
        g.add_edge(a, d)
        g.add_edge(c, b)

        # update our edge list so we don't lose track
        edges[idx1] = (a, d)
        edges[idx2] = (c, b)
        swaps_done += 1

    return g

# ============================================================
# GENERATE A BUNCH AND SEE WHAT HAPPENS
# ============================================================
num_instances = 100
max_k = max(degrees) + 1
degree_vals = np.arange(max_k)

# accumulators for the probability distribution at each degree
cm_avg_pdf = np.zeros(max_k)
esw_avg_pdf = np.zeros(max_k)

def tally_degrees(g, pdf_accumulator):
    """count up degrees and add to the probability accumulator"""
    degs = [d for _, d in g.degree()]
    cnt = Counter(degs)
    n = g.number_of_nodes()
    for k, c in cnt.items():
        if k < max_k:
            pdf_accumulator[k] += c / n

print("Generating 100 Configuration Model instances...")
for i in range(num_instances):
    g_cm = make_graph_from_degrees(degrees)
    tally_degrees(g_cm, cm_avg_pdf)
    if (i + 1) % 20 == 0:
        print(f"  CM: {i+1}/100 done")

cm_avg_pdf /= num_instances  # average it out

print("Generating 100 Edge-Swapping instances...")
for i in range(num_instances):
    g_esw = rewire_it(real_g)
    tally_degrees(g_esw, esw_avg_pdf)
    if (i + 1) % 20 == 0:
        print(f"  ESW: {i+1}/100 done")

esw_avg_pdf /= num_instances  # average it out

# real graph degree distribution for comparison
real_counts = Counter(degrees)
real_pdf = np.array([real_counts.get(k, 0) / real_g.number_of_nodes()
                      for k in degree_vals])

# ============================================================
# PICTURE TIME: LET'S SEE THESE DISTRIBUTIONS
# ============================================================
mask = real_pdf > 0  # only plot degrees that actually exist

fig, ax = plt.subplots(figsize=(9, 5))

ax.loglog(degree_vals[mask], real_pdf[mask],  'bo-', ms=4, lw=1.5,
          label='Original (Facebook)')
ax.loglog(degree_vals[mask], cm_avg_pdf[mask],     'rs--', ms=4, lw=1.5,
          label='Configuration Model (avg 100)')
ax.loglog(degree_vals[mask], esw_avg_pdf[mask],    'g^-.', ms=4, lw=1.5,
          label='Edge-Swapping (avg 100)')

ax.set_xlabel('Degree  $k$', fontsize=13)
ax.set_ylabel('$P(k)$',      fontsize=13)
ax.set_title('Degree Distribution: Original vs Randomised Graphs\n(Facebook Network)',
             fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig('Q1_degree_distribution.png', dpi=150)
plt.show()
print("Plot saved as Q1_degree_distribution.png")

# ============================================================
# SOME BASIC STATS ABOUT THE REAL NETWORK
# ============================================================
def get_stats(g, degree_sequence):
    """grab the usual suspects: node/edge counts, degree stats, connectivity"""
    print(f"  Nodes           : {g.number_of_nodes()}")
    print(f"  Edges           : {g.number_of_edges()}")
    print(f"  Avg Degree      : {np.mean(degree_sequence):.3f}")
    print(f"  Max Degree      : {max(degree_sequence)}")
    print(f"  Min Degree      : {min(degree_sequence)}")
    print(f"  Is Connected    : {nx.is_connected(g)}")
    print(f"  Avg Clustering  : {nx.average_clustering(g):.4f}")

print("\n=== Real Network Properties ===")
get_stats(real_g, degrees)
